In [ ]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [ ]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [ ]:
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": [
        "surfline.com",        # forecasts + board reviews, industry standard
        "boardcave.com",       # board/quiver finder + volume guidance
        "stabmag.com",         # board reviews, surf media
        "theinertia.com",      # gear/board reviews
        "channelislands.com",  # shaper site, board specs + volume
        "lostsurfboards.com",  # shaper site, board specs + volume
        "firewiresurfboards.com",  # shaper site, specs + volume
        "jsindustries.com",    # shaper site, specs + volume
    ],
}

In [ ]:
messages = []
add_user_message(
    messages,
    """
    Antes de responder, busca en la web modelos y litrajes actuales de tablas
    (mid-lengths, funboards, shortboards) usando las fuentes disponibles, y cita
    las fuentes que uses.

    Qué tablas de surf me convienen usar? cómo armarías mi quiver? tengo 39 años, mido 1.75mts, peso 75kg. 
    Vivo en Miraflores, Lima, Perú y surfeo en Makaha, porque es la playa más cercana que tengo.
    Esta playa tiene olas gordas, desordenadas, mucha espuma y hay que remar 10min o más para llegar al point atravesando espuma.
    En San Bartolo, Lima, Perú, donde viví por 10 años y queda la casa de mi madre a la que voy los fines de semana cada 2 semanas,
    tengo mi longboard 9,2 de epoxica piccolo clemente y una 6,2 tabla corta 
    en Miraflores tengo una 6,6 erik arakawa de 30lts con la que me cuesta remar y pararme
    suelo surfear mejor con mi longboard y en san bartolo
    Qué tablas de surf me convienen usar? cómo armarías mi quiver? compara ventajas y desventajas y dame mi quiver armado y
    qué tablas debería tener y dónde
    """,
)
response = chat(messages, tools=[web_search_schema])
response

In [ ]:
from IPython.display import Markdown, display

def show(message):
    parts = []
    for block in message.content:
        if block.type == "text":
            parts.append(block.text)
        elif block.type == "tool_use" or block.type == "server_tool_use":
            parts.append(f"\n> 🔧 **{block.name}** called with `{block.input}`\n")
    display(Markdown("\n".join(parts)))
    print(f"\n— stop_reason: {message.stop_reason} | "
        f"in: {message.usage.input_tokens} tok, out: {message.usage.output_tokens} tok")

show(response)